# MATH 189 Project: The Sound of Inequality

This notebook will serve as a list of all our data sources.

---

## Datasets

1. 
2. Zillow ZHVI ("Typical home value for the region") by ZIP
3. ZIP to ZCTA Conversion
4. 2020 Population Demographics Census
5. American Community Survey Housing Data
6. American Community Survey Economics Data

In [13]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import zipfile
import re

### (1) National Transportation Noise Map by ? (need to figure how to load in)

From the Bureau of Transportation Statistics

Link to documentation: https://rosap.ntl.bts.gov/view/dot/53773

In [19]:
!pip install "numpy<2.0" rasterio geopandas requests --force-reinstall

import requests
import rasterio
from rasterio.io import MemoryFile
import numpy as np
import pandas as pd

# ① 从 BTS API 下载 San Diego 噪音栅格图
url = "https://maps.bts.dot.gov/arcgis/rest/services/NTMS/NationalTransportationNoiseMap_2020/ImageServer/exportImage"

params = {
    "bbox": "-117.6,32.5,-116.1,33.4",  # San Diego 经纬度范围
    "bboxSR": "4326",
    "imageSR": "4326",
    "size": "1000,800",
    "format": "tiff",
    "pixelType": "F32",
    "f": "image"
}

response = requests.get(url, params=params)
print("状态码:", response.status_code)
print("文件大小:", len(response.content), "bytes")

# ② 读取返回的 GeoTIFF（不需要存到硬盘）
with MemoryFile(response.content) as memfile:
    with memfile.open() as dataset:
        noise_array = dataset.read(1)   # 读取噪音值（dB）
        transform = dataset.transform
        crs = dataset.crs
        print("图像尺寸:", noise_array.shape)
        print("CRS:", crs)
        
        # 过滤掉无效值（BTS用-9999表示无数据）
        valid = noise_array[noise_array > 0]
        print("噪音值范围: {:.1f} ~ {:.1f} dB".format(valid.min(), valid.max()))
        print("平均噪音: {:.1f} dB".format(valid.mean()))

  Using cached rasterio-1.5.0-cp312-cp312-macosx_14_0_arm64.whl.metadata (8.6 kB)
  Using cached geopandas-1.1.3-py3-none-any.whl.metadata (2.3 kB)
  Using cached affine-2.4.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached cligj-0.7.2-py3-none-any.whl.metadata (5.0 kB)
INFO: pip is looking at multiple versions of rasterio to determine which version is compatible with other requirements. This could take a while.
  Using cached pyogrio-0.12.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (5.9 kB)
  Using cached pyproj-3.7.2-cp312-cp312-macosx_14_0_arm64.whl.metadata (31 kB)
  Using cached shapely-2.1.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 43.1 MB/s eta 0:00:001m35.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 15.1 MB/s eta 0:00:0031m15.4 MB/s eta 0:00:01
Using cached geopandas-1.1.3-py3-none-any.whl (342 kB)
Using cached cligj-0.7.2-py3-none-any.whl (7.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

ConnectionError: HTTPSConnectionPool(host='maps.bts.dot.gov', port=443): Max retries exceeded with url: /arcgis/rest/services/NTMS/NationalTransportationNoiseMap_2020/ImageServer/exportImage?bbox=-117.6%2C32.5%2C-116.1%2C33.4&bboxSR=4326&imageSR=4326&size=1000%2C800&format=tiff&pixelType=F32&f=image (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x135c959a0>: Failed to resolve 'maps.bts.dot.gov' ([Errno 8] nodename nor servname provided, or not known)"))

### (2) Zillow ZHVI ("Typical home value for the region") by ZIP

From Zillow Research

Link to documentation on ZHVI: https://www.zillow.com/research/zhvi-user-guide/

In [3]:
# Load in dataset
zillow_home_value_data = pd.read_csv('zillow_zip_zhvi.csv')
zillow_home_value_data

,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,2000-01-31,...,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30
0,91982,1,77494,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Fort Bend County,211041.005329,...,493611.981869,492490.195382,492692.837484,493445.443112,494524.551492,495131.184301,494732.516894,493303.297817,491671.876271,489900.281466
1,61148,2,8701,zip,NJ,NJ,Lakewood,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,113107.940019,...,543090.959009,544341.283601,546905.912540,551821.486041,557770.333985,564044.483445,568535.788174,572276.269997,575223.660538,577895.151126
2,91940,3,77449,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Harris County,104955.931168,...,277240.630382,276294.644916,275506.866356,274758.533310,274115.630288,273772.685442,273217.060982,272811.268664,272413.963259,272122.210142
3,62080,4,11368,zip,NY,NY,New York,"New York-Newark-Jersey City, NY-NJ-PA",Queens County,168141.944305,...,518740.000548,520355.644223,521555.324508,522550.762047,524410.419569,527020.920312,531651.968961,536447.865586,541178.231366,543161.437084
4,91733,5,77084,zip,TX,TX,Houston,"Houston-The Woodlands-Sugar Land, TX",Harris County,104704.775938,...,273656.086467,272762.684406,271867.517224,271000.272267,270256.217132,269906.006551,269391.827130,268904.230398,268418.343637,267990.471321
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26271,63828,39992,15006,zip,PA,PA,Bairdford,"Pittsburgh, PA",Allegheny County,NaN,...,181426.652692,183242.701103,185103.330714,186806.434482,187131.014252,186675.444439,185483.787820,184959.617499,184667.600181,184062.236413
26272,64510,39992,16238,zip,PA,PA,Manorville,"Pittsburgh, PA",Armstrong County,NaN,...,119337.145471,117006.575389,114163.043112,110992.459729,107784.627991,104926.749482,102628.710625,101159.768282,99588.937191,99543.248022
26273,98388,39992,95697,zip,CA,CA,Yolo,"Sacramento-Roseville-Folsom, CA",Yolo County,NaN,...,468222.616955,469487.818430,472449.460158,473808.740111,475843.329571,480240.037672,486055.938729,491404.104545,493170.615388,493680.685695
26274,72138,39992,32664,zip,FL,FL,Micanopy,"Ocala, FL",Marion County,NaN,...,300444.282356,297357.147543,295847.355144,295253.680368,295896.983004,296405.407181,297179.743004,298957.163110,301912.175312,304760.230942


### (3) ZIP to ZCTA Conversion

Comparing a ZCTA (ZIP Code Tabulation Area) to a USPS ZIP Code means comparing a statistical geographic polygon (the Census Bureau's estimate of where a ZIP code lives) with a linear postal delivery route. ZIP codes are used by USPS and are changed frequently, causing issues because they include PO Boxes, corporate campuses, military bases.

We should be using ZCTA as a more accurate reflection of area and go by Census standards.

This is necessary for dataset #2 and any other additional datasets that use ZIP codes.

Link to documentation on how the conversion was done: https://github.com/censusreporter/acs-aggregate/blob/master/crosswalks/zip_to_zcta/build_crosswalk.ipynb

In [4]:
# Load in dataset
zip_zcta_conversion_data = pd.read_csv('zip_zcta_xref.csv')
zip_zcta_conversion_data

,zip_code,zcta,source
0,99553,99553.0,geonames
1,99571,99571.0,geonames
2,99583,99583.0,geonames
3,99612,99612.0,geonames
4,99661,99661.0,geonames
...,...,...,...
41126,979,979.0,tiger
41127,982,982.0,tiger
41128,66019,66019.0,tiger
41129,98929,98929.0,tiger


### (4) 2020 Census Data ("National Neighborhood Archive) by ZIP

Contains information on area, population, race, language, education, family income, marriage status, age, employment, poverty, median family income. 

NOTE! This can also be pulled from the same website as the follow list (under DP01 and DP02) but this is a more 'condensed/aggregated version'

Link to source: https://www.icpsr.umich.edu/web/ICPSR/studies/38528/summary

In [25]:
# Load in dataset
rows = []

with zipfile.ZipFile('ICPSR_38528-V6.zip') as z:
    with z.open('ICPSR_38528/DS0008/38528-0008-Data.txt') as f:

        for line in f:

            # convert bytes -> string
            line = line.decode("utf-8").strip()

            # skip empty rows
            if not line:
                continue

            # split on whitespace
            parts = re.split(r"\s+", line)

            rows.append(parts)

# Find maximum number of columns
max_cols = max(len(r) for r in rows)

# Pad short rows
rows = [r + [None]*(max_cols - len(r)) for r in rows]

# Create dataframe
census_demographics_data = pd.DataFrame(rows)

# Preserve ZIP codes
census_demographics_data[0] = census_demographics_data[0].astype(str).str.zfill(5)

census_demographics_data

,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,38
0,00601,64.420341492,16834,261.31497192,.9935843944550,.0042176549323,.0010098610073,.0027919686399,.7401638031006,.3141927719116,...,.7002434134483,.1416722685099,.6760179996490,.5788466930389,19628,22969,19495,None,None,None
1,00602,30.327054977,37642,1241.20190430,.9459646344185,.0437277518213,.0000265660692,.0071728387848,.6671696901321,.3325094580650,...,.7536980509758,.1561276614666,.5973837375641,.5401023626328,24126,23625,23922,None,None,None
2,00603,34.346618652,49075,1428.81604004,.9809678792953,.0139378504828,.0006113091949,.0127967400476,.5840433835983,.2432864159346,...,.5736852884293,.1970510631800,.5702557563782,.5259360074997,24062100588,20156,23758,4.990474224091,4.233858108521,None
3,00606,44.334327698,5590,126.08739471,.9932021498680,.0067978533916,.0000000000000,.0085867624730,.8562940955162,.4050509929657,...,.7510266900063,.0649080500007,.6358803510666,.6193609833717,21439,21345,None,None,None,None
4,00610,37.115749359,25542,688.17144775,.9634719491005,.0293242502958,.0047764466144,.0055203195661,.7055690288544,.2755567133427,...,.7279353141785,.1564851403236,.5688122510910,.5581871271133,26914,27417,26780,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33769,99923,17.343692780,25,1.44144619,.00000000000001.0000000000000,.0000000000000,.0000000000000,.0000000000000,.0000000000000,.3600000143051,...,None,None,None,None,None,None,None,None,None,None
33770,99925,53.626453400,920,17.15571213,.0608695633709,.4250000119209,.0043478258885,.0315217375755,.0078387456015,.0968703404069,...,.7326202988625,.2847226858139,.1477056890726,.0334100164473,91042134286,None,None,None,None,None
33771,99926,132.797744751,1465,11.03181362,.0368600673974,.0737201347947,.0034129691776,.0259385667741,.0057306592353,.0774719640613,...,.7762863636017,.2613121271133,.2272387593985,.0228430982679,95278181000,None,None,None,None,None
33772,99927,4.679345131,14,2.99187160,.00000000000001.0000000000000,.00000000000001.0000000000000,.0000000000000,.00000000000001.0000000000000,.0000000000000,.0000000000000,...,None,None,None,None,None,None,None,None,None,None


### (5) American Community Survey Housing Data by ZCTA

Also from the bureau but this is a 5 year estimate on housing numbers. Includes Occupancy and Structure, Housing Value and Costs, Utilities.

Link to Source: https://www.census.gov/acs/www/data/data-tables-and-tools/data-profiles/

In [16]:
# Load in dataset from zip folder
with zipfile.ZipFile('ACSDP5Y2024.DP04_2026-05-25T014331.zip') as z:
    with z.open('ACSDP5Y2024.DP04-Data.csv') as f:
        acs_housing_data = pd.read_csv(f)

acs_housing_data

C:\Users\Ivan Chen\AppData\Local\Temp\ipykernel_63652\3125503151.py:4: DtypeWarning: Columns (2,3,4,5,6,7,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,204,205,206,207,208,209,210,211,212,213,214,215,216,217,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,

,GEO_ID,NAME,DP04_0001E,DP04_0001M,DP04_0002E,DP04_0002M,DP04_0003E,DP04_0003M,DP04_0004E,DP04_0004M,...,DP04_0139PM,DP04_0140PE,DP04_0140PM,DP04_0141PE,DP04_0141PM,DP04_0142PE,DP04_0142PM,DP04_0143PE,DP04_0143PM,Unnamed: 574
0,Geography,Geographic Area Name,Estimate!!HOUSING OCCUPANCY!!Total housing units,Margin of Error!!HOUSING OCCUPANCY!!Total hous...,Estimate!!HOUSING OCCUPANCY!!Total housing uni...,Margin of Error!!HOUSING OCCUPANCY!!Total hous...,Estimate!!HOUSING OCCUPANCY!!Total housing uni...,Margin of Error!!HOUSING OCCUPANCY!!Total hous...,Estimate!!HOUSING OCCUPANCY!!Total housing uni...,Margin of Error!!HOUSING OCCUPANCY!!Total hous...,...,Percent Margin of Error!!GROSS RENT AS A PERCE...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,Percent!!GROSS RENT AS A PERCENTAGE OF HOUSEHO...,Percent Margin of Error!!GROSS RENT AS A PERCE...,NaN
1,860Z200US00601,ZCTA5 00601,7570,322,5768,303,1802,253,0.4,0.6,...,8.8,8.5,5.5,11.5,6.6,38.0,10.4,(X),(X),NaN
2,860Z200US00602,ZCTA5 00602,17610,492,12954,592,4656,393,0.3,0.3,...,8.0,8.7,5.8,10.0,7.0,32.2,10.7,(X),(X),NaN
3,860Z200US00603,ZCTA5 00603,26239,671,20131,780,6108,460,0.8,0.5,...,3.3,18.4,6.1,9.3,3.4,35.0,6.2,(X),(X),NaN
4,860Z200US00606,ZCTA5 00606,2681,196,1860,182,821,153,0.0,3.6,...,8.0,0.0,29.5,11.2,13.1,25.0,18.3,(X),(X),NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33768,860Z200US99923,ZCTA5 99923,53,34,18,18,35,32,0.0,61.3,...,**,-,**,-,**,-,**,(X),(X),NaN
33769,860Z200US99925,ZCTA5 99925,440,50,387,51,53,15,0.0,9.0,...,4.0,13.1,9.3,11.2,7.1,19.6,10.3,(X),(X),NaN
33770,860Z200US99926,ZCTA5 99926,538,43,475,49,63,21,0.0,7.4,...,9.4,3.1,5.2,0.0,17.3,17.6,11.3,(X),(X),NaN
33771,860Z200US99927,ZCTA5 99927,36,36,27,34,9,12,0.0,50.1,...,**,-,**,-,**,-,**,(X),(X),NaN


### (6) American Community Survey Economics Data by ZCTA

Also from the bureau but this is a 5 year estimate on economics numbers. Includes Income, Employment, Occupation, Commuting to Work.

Link to Source: https://www.census.gov/acs/www/data/data-tables-and-tools/data-profiles/

In [15]:
# Load in dataset from zip folder
with zipfile.ZipFile('ACSDP5Y2024.DP03_2026-05-25T014948.zip') as z:
    with z.open('ACSDP5Y2024.DP03-Data.csv') as f:
        acs_economics_data = pd.read_csv(f)

acs_economics_data

C:\Users\Ivan Chen\AppData\Local\Temp\ipykernel_63652\3672969644.py:4: DtypeWarning: Columns (2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,128,129,132,133,136,137,140,141,144,145,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,178,179,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,276,290,294,302,306,310,326,338,366,376,424,452,464,474,478,480,482,492,502) have mixed types. Specify dtype option on import or set low_memory=False.
  acs_economics_data = pd.read_csv(f)


,GEO_ID,NAME,DP03_0001E,DP03_0001M,DP03_0002E,DP03_0002M,DP03_0003E,DP03_0003M,DP03_0004E,DP03_0004M,...,DP03_0133PM,DP03_0134PE,DP03_0134PM,DP03_0135PE,DP03_0135PM,DP03_0136PE,DP03_0136PM,DP03_0137PE,DP03_0137PM,Unnamed: 550
0,Geography,Geographic Area Name,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,Estimate!!EMPLOYMENT STATUS!!Population 16 yea...,Margin of Error!!EMPLOYMENT STATUS!!Population...,...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,Percent!!PERCENTAGE OF FAMILIES AND PEOPLE WHO...,Percent Margin of Error!!PERCENTAGE OF FAMILIE...,NaN
1,860Z200US00601,ZCTA5 00601,14094,463,5913,510,5913,510,4698,468,...,5.0,58.8,5.8,50.2,6.8,58.8,5.3,65.7,5.9,NaN
2,860Z200US00602,ZCTA5 00602,32580,277,12603,834,12603,834,11956,788,...,3.7,44.9,4.5,44.2,4.6,44.8,4.3,57.7,5.3,NaN
3,860Z200US00603,ZCTA5 00603,41848,877,17050,979,17018,982,14405,912,...,2.6,42.7,3.4,38.7,3.6,40.3,3.1,61.1,3.8,NaN
4,860Z200US00606,ZCTA5 00606,4436,259,1533,237,1533,237,1533,237,...,8.2,56.9,9.1,41.8,13.3,52.8,9.2,61.5,12.4,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33768,860Z200US99923,ZCTA5 99923,24,26,18,18,18,18,18,18,...,53.1,-,**,0.0,53.1,0.0,69.5,0.0,82.3,NaN
33769,860Z200US99925,ZCTA5 99925,691,83,417,60,417,60,394,58,...,4.2,18.4,5.2,5.8,4.3,11.6,5.3,31.6,9.4,NaN
33770,860Z200US99926,ZCTA5 99926,1069,148,702,83,702,83,653,84,...,8.9,15.1,9.0,9.8,9.8,12.3,9.3,19.5,7.6,NaN
33771,860Z200US99927,ZCTA5 99927,27,34,0,10,0,10,0,10,...,50.1,0.0,50.1,-,**,-,**,0.0,50.1,NaN
